In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, f1_score
import mlflow
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = spark.table("dataanalytics.ml1.loan_book_gold_v2").toPandas()

# Split data using existing data_split column
train_df = df[df['data_split'] == 'train'].copy()
test_df = df[df['data_split'] == 'test'].copy()

# Separate features and target
y_train = train_df['default_flag']
y_test = test_df['default_flag']

# Drop target and split column to get features
feature_cols = [col for col in df.columns if col not in ['default_flag', 'data_split']]
X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

# WoE Encoding Implementation
class WoEEncoder:
    def __init__(self, n_bins=10):
        self.n_bins = n_bins
        self.woe_mapping = {}
        
    def fit(self, X, y):
        """Fit WoE encoder on training data"""
        for col in X.columns:
            # Create bins using quantiles
            X_col = pd.to_numeric(X[col], errors='coerce')
            try:
                bins = pd.qcut(X_col, q=self.n_bins, duplicates='drop', retbins=True)[1]
            except:
                bins = pd.cut(X_col, bins=self.n_bins, duplicates='drop', retbins=True)[1]
            
            bins[0] = -np.inf
            bins[-1] = np.inf
            
            # Bin the values
            binned = pd.cut(X_col, bins=bins)
            
            # Calculate WoE for each bin
            woe_dict = {}
            total_good = (y == 0).sum()
            total_bad = (y == 1).sum()
            
            for bin_val in binned.cat.categories:
                mask = binned == bin_val
                n_good = ((y[mask] == 0).sum())
                n_bad = ((y[mask] == 1).sum())
                
                # Add small constant to avoid division by zero
                n_good = max(n_good, 0.5)
                n_bad = max(n_bad, 0.5)
                
                pct_good = n_good / total_good
                pct_bad = n_bad / total_bad
                
                woe = np.log(pct_good / pct_bad)
                woe_dict[bin_val] = woe
            
            self.woe_mapping[col] = {'bins': bins, 'woe': woe_dict}
        
        return self
    
    def transform(self, X):
        """Transform features using fitted WoE mapping"""
        X_woe = pd.DataFrame(index=X.index)
        
        for col in X.columns:
            if col in self.woe_mapping:
                bins = self.woe_mapping[col]['bins']
                woe_dict = self.woe_mapping[col]['woe']
                
                # Bin the values
                X_col = pd.to_numeric(X[col], errors='coerce')
                binned = pd.cut(X_col, bins=bins)
                
                # Map to WoE values - convert to list first to avoid categorical issues
                woe_values = [woe_dict.get(bin_val, 0) if pd.notna(bin_val) else 0 
                             for bin_val in binned]
                X_woe[col] = woe_values
        
        return X_woe
    
    def fit_transform(self, X, y):
        """Fit and transform in one step"""
        self.fit(X, y)
        return self.transform(X)

# Fit WoE encoder on training data only
woe_encoder = WoEEncoder(n_bins=10)
X_train = woe_encoder.fit_transform(X_train_raw, y_train)

# Transform test data using fitted encoder
X_test = woe_encoder.transform(X_test_raw)

# Train Logistic Regression model with MLflow tracking
mlflow.set_experiment("/Users/prosperproper700@gmail.com/credit_scorecard")

with mlflow.start_run(run_name="logistic_scorecard_woe"):
    # Train model
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    # Generate predictions on test set
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    auc = roc_auc_score(y_test, y_pred_proba)
    gini = 2 * auc - 1
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Store metrics in dictionary
    metrics_dict = {
        'AUC': auc,
        'Gini Coefficient': gini,
        'Precision': precision,
        'Recall': recall,
        'Accuracy': accuracy,
        'F1-score': f1
    }
    
    # Log to MLflow
    mlflow.log_metrics(metrics_dict)
    mlflow.sklearn.log_model(model, "logistic_model")
    
    # Extract model equation
    intercept = model.intercept_[0]
    coefficients = model.coef_[0]
    feature_names = X_train.columns
    
    # Build equation string
    equation_parts = [f"η = {intercept:.6f}"]
    for feat, coef in zip(feature_names, coefficients):
        sign = "+" if coef >= 0 else ""
        equation_parts.append(f"{sign}{coef:.6f} × WoE({feat})")
    
    equation = " ".join(equation_parts)
    
    # Store results
    results = {
        'metrics': metrics_dict,
        'equation': equation,
        'intercept': intercept,
        'coefficients': dict(zip(feature_names, coefficients))
    }

display(pd.DataFrame([metrics_dict]))
equation

2026/05/18 19:15:23 INFO mlflow.tracking.fluent: Experiment with name '/Users/prosperproper700@gmail.com/credit_scorecard' does not exist. Creating a new experiment.
2026/05/18 19:15:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-4225c63b-638a.cloud.databricks.com/ml/experiments/2914302312385326/models/m-bdbf055f485249099ca8a612d696ba92?o=7474645869796416
2026/05/18 19:15:29 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


AUC,Gini Coefficient,Precision,Recall,Accuracy,F1-score
0.7799175899047945,0.5598351798095891,0.5957854406130269,0.16798703637018364,0.854560553633218,0.26207865168539324


'η = -1.704449 -0.595744 × WoE(annual_income) -0.326947 × WoE(interest_rate) -0.633178 × WoE(employment_length_years) -0.287620 × WoE(num_delinquencies_2yr) -0.680532 × WoE(months_since_oldest_account) -1.181109 × WoE(num_open_accounts) -0.545649 × WoE(loan_amount) +0.038100 × WoE(dti_ratio) -0.641392 × WoE(months_since_last_delinquency) -0.344233 × WoE(pct_accounts_current) +0.000000 × WoE(high_earner_low_stability_flag) -0.431476 × WoE(loan_to_income_ratio) -0.183723 × WoE(debt_servicing_stress_index) -0.391996 × WoE(delinquency_concentration)'